In [0]:
storage_account_name = "cloudproject60306117"
storage_account_key = "AdxIWpu+RCT/r8Gr8rvVZxi25XMS/T0QnFugMQMzWXp6P7RHljEIpEZooyARZ3VJAEalcNRoGaDu+AStblz2Bw=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
from pyspark.sql.functions import col, hour, dayofweek, when

# Load sampled dataset
sampled_path = "abfss://curated@cloudproject60306117.dfs.core.windows.net/features_v1_sampled/"
df = spark.read.parquet(sampled_path)

# Add engineered features
df = df.withColumn("is_weekend", when(col("pickup_day_of_week").isin([6,7]), 1).otherwise(0))
df = df.withColumn("avg_speed_mph", 
                   when(col("trip_duration_minutes") > 0, 
                        (col("trip_distance") / (col("trip_duration_minutes") / 60))
                   ).otherwise(0))

# Save engineered features
features_path = "abfss://curated@cloudproject60306117.dfs.core.windows.net/features_v2/"
df.write.mode("overwrite").parquet(features_path)

print("Feature engineered dataset rows:", df.count())


Feature engineered dataset rows: 300000
